# 02 — Bias Analysis

**NovaCred Credit Application Governance | DEGO 2606**

**Author:** Justus Nau (Data Scientist)

---

## Objectives

Detect bias and fairness issues in NovaCred's historical credit decisions:

| Analysis | Description |
|----------|-------------|
| Disparate Impact | Gender-based approval rate disparity (four-fifths rule) |
| Proxy Discrimination | Non-protected attributes correlating with protected characteristics |
| Age-Based Patterns | Approval and interest rate patterns across age groups |
| Interaction Effects | Combined effects of multiple protected attributes |

---

## Sections

0. Setup & Data Loading
1. Disparate Impact Analysis (Gender)
2. Age-Based Bias Patterns
3. Proxy Discrimination Analysis
4. Interaction Effects
5. Summary & Fairness Recommendations

---
## Section 0 — Setup & Data Loading

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project root from notebooks/ directory
project_root = Path.cwd().resolve().parent
data_path = project_root / "data" / "clean_credit_applications.csv"

# Load cleaned dataset
df = pd.read_csv(data_path)

# Quick sanity checks
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Loaded: /Users/justus/Desktop/Data Governance Team Project/dego-project-team5/data/clean_credit_applications.csv
Shape: 502 rows × 16 columns

Columns:
['_id', 'gender', 'date_of_birth', 'zip_code', 'annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance', 'loan_approved', 'interest_rate', 'approved_amount', 'rejection_reason', 'loan_purpose', 'notes', 'processing_timestamp', 'total_flags']


,_id,gender,date_of_birth,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,loan_purpose,notes,processing_timestamp,total_flags
0,app_200,Male,2001-03-09,10036.0,73000.0,23,0.20,31212,False,NaN,NaN,algorithm_risk_score,NaN,NaN,2024-01-15T00:00:00Z,0
1,app_037,Male,1992-03-31,10032.0,78000.0,51,0.18,17915,False,NaN,NaN,algorithm_risk_score,NaN,NaN,NaN,1
2,app_215,Male,1989-10-24,10075.0,61000.0,41,0.21,37909,True,3.7,59000.0,NaN,vacation,NaN,NaN,0
3,app_024,Male,1983-04-25,10077.0,103000.0,70,0.35,0,True,4.3,34000.0,NaN,NaN,NaN,NaN,0
4,app_184,Male,1999-05-21,10080.0,57000.0,14,0.23,31763,False,NaN,NaN,algorithm_risk_score,NaN,NaN,2024-01-15T00:00:00Z,1


---
## Section 1 — Disparate Impact Analysis (Gender)

In [3]:
# Approval rate by gender
gender_rates = (
    df.dropna(subset=['gender', 'loan_approved'])
      .groupby('gender')['loan_approved']
      .mean()
      .sort_values(ascending=False)
      .rename('approval_rate')
)

gender_summary = (gender_rates * 100).round(2).to_frame('approval_rate_pct')
print('Approval rate by gender (%):')
print(gender_summary.to_string())

Approval rate by gender (%):
        approval_rate_pct
gender                   
Male                65.73
Female              50.60


In [4]:
# Disparate Impact (DI) ratio and four-fifths rule interpretation
privileged_gender = gender_rates.idxmax()
unprivileged_gender = gender_rates.idxmin()

approval_privileged = gender_rates.loc[privileged_gender]
approval_unprivileged = gender_rates.loc[unprivileged_gender]

di_ratio = approval_unprivileged / approval_privileged

print(f'Privileged group (higher approval rate): {privileged_gender} ({approval_privileged:.4f})')
print(f'Unprivileged group (lower approval rate): {unprivileged_gender} ({approval_unprivileged:.4f})')
print(f'\nDisparate Impact Ratio (DI) = {di_ratio:.4f}')

threshold = 0.80
if di_ratio < threshold:
    print('Interpretation: DI < 0.80 → potential disparate impact (four-fifths rule violated).')
else:
    print('Interpretation: DI >= 0.80 → no disparate impact flagged by the four-fifths rule.')

Privileged group (higher approval rate): Male (0.6573)
Unprivileged group (lower approval rate): Female (0.5060)

Disparate Impact Ratio (DI) = 0.7698
Interpretation: DI < 0.80 → potential disparate impact (four-fifths rule violated).


In [ ]:
# Bar chart of approval rates by gender with DI annotation
import matplotlib.pyplot as plt

plot_df = gender_rates.sort_values(ascending=False).mul(100).reset_index()
plot_df.columns = ['gender', 'approval_rate_pct']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(plot_df['gender'], plot_df['approval_rate_pct'])

for bar in bars:
    value = bar.get_height()
    ax.annotate(
        f'{value:.1f}%',
        xy=(bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, 5),
        textcoords='offset points',
        ha='center',
        va='bottom'
    )

ax.set_title(f'Approval Rate by Gender (DI = {di_ratio:.3f})')
ax.set_xlabel('Gender')
ax.set_ylabel('Approval Rate (%)')
ax.set_ylim(0, max(plot_df['approval_rate_pct']) * 1.2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/bias_approval_rate_by_gender.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2 — Age-Based Bias Patterns